# 03 — Kraken : tel quel, puis affiné

Modèle de départ : [CATMuS Medieval 1.5.0](https://doi.org/10.5281/zenodo.12743230), Kraken 7.1.1.

| Modèle | CER test |
|---|---|
| Tel quel, lignes découpées par rectangle | 40,1 % |
| Tel quel, lignes découpées par polygone | 16,7 % |
| Affiné sur le train HTRogène | **7,2 %** |

**Environnement** : Google Colab avec **GPU T4** (l'évaluation seule tourne aussi sur CPU). Ce notebook doit rester séparé des notebooks Qwen : Kraken impose une version de `safetensors` incompatible avec `transformers`, qu'on désinstalle ici.

In [ ]:
!pip install -q kraken jiwer
!pip uninstall -y -q transformers          # conflit safetensors : inutile ici
!kraken get 10.5281/zenodo.12743230

import json, csv, shutil, subprocess, unicodedata
from pathlib import Path
from collections import defaultdict
import jiwer

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/projet_vlm")
except ImportError:
    ROOT = Path("projet_vlm")

MODEL_BASE = next(Path("/root/.local/share/htrmopo").rglob("catmus-medieval.mlmodel"))
print(MODEL_BASE)

## Fonction d'évaluation

Chaque image de ligne est transcrite par `kraken ocr --no-segmentation` (l'image est déjà une ligne). CER calculé sur l'ensemble du test avec `jiwer`, après normalisation **NFD** des deux côtés (normalisation d'entraînement du modèle CATMuS).

In [ ]:
def nfd(s):
    return unicodedata.normalize("NFD", s or "").strip()

def evaluer(donnees, modele, nom_csv):
    tout = [json.loads(l) for l in open(donnees / "lignes.jsonl", encoding="utf-8")]
    test = [e for e in tout if e["split"] == "test"]
    w = Path("eval_" + nom_csv.replace(".csv", ""))
    shutil.rmtree(w, ignore_errors=True); w.mkdir()
    for e in test:
        shutil.copy(donnees / e["image"], w / Path(e["image"]).name)
    subprocess.run(["kraken", "-I", f"{w}/*.png", "-o", ".txt", "ocr", "-m", str(modele), "--no-segmentation"],
                   check=True, capture_output=True)

    vrais, preds, par_ms = [], [], defaultdict(lambda: ([], []))
    for e in test:
        nom = Path(e["image"]).name.replace(".png", "")
        sorties = list(w.glob(nom + "*.txt"))
        p = nfd(sorties[0].read_text(encoding="utf-8")) if sorties else ""
        v = nfd(e["text"])
        vrais.append(v); preds.append(p)
        par_ms[e["manuscrit"]][0].append(v); par_ms[e["manuscrit"]][1].append(p)

    with open(ROOT / nom_csv, "w", encoding="utf-8", newline="") as f:
        wr = csv.writer(f); wr.writerow(["image", "manuscrit", "vrai", "pred"])
        for e, v, p in zip(test, vrais, preds):
            wr.writerow([e["image"], e["manuscrit"], v, p])

    print(f"CER global : {jiwer.cer(vrais, preds):.1%}")
    for m, (v, p) in par_ms.items():
        print(f"  {m:25} {len(v):4} lignes   CER {jiwer.cer(v, p):.1%}   vides : {sum(1 for x in p if not x)}")

## 1. Kraken tel quel

In [ ]:
if (ROOT / "lignes_occitan").exists():                       # découpage par rectangle (optionnel)
    print("Rectangle :"); evaluer(ROOT / "lignes_occitan", MODEL_BASE, "predictions_kraken.csv")
print("Polygone :");  evaluer(ROOT / "lignes_occitan_poly", MODEL_BASE, "predictions_kraken_poly.csv")

## 2. Affinage sur le train

Format `path` : chaque image de ligne accompagnée d'un fichier `.gt.txt` contenant sa transcription. Le test n'est jamais utilisé à l'entraînement ; le meilleur checkpoint est choisi sur le dev.

In [ ]:
DATA = ROOT / "lignes_occitan_poly"
tout = [json.loads(l) for l in open(DATA / "lignes.jsonl", encoding="utf-8")]
KDIR = Path("kraken_data"); KDIR.mkdir(exist_ok=True)
listes = {"train": [], "dev": []}
for e in tout:
    if e["split"] in listes:
        img = KDIR / Path(e["image"]).name
        shutil.copy(DATA / e["image"], img)
        img.with_suffix(".gt.txt").write_text(e["text"], encoding="utf-8")
        listes[e["split"]].append(str(img.resolve()))
for split, chemins in listes.items():
    Path(f"{split}.txt").write_text("\n".join(chemins), encoding="utf-8")
    print(split, len(chemins))        # train 3392, dev 327

Réglages utilisés pour les résultats publiés : extension de l'alphabet aux caractères absents de CATMuS (`--resize union`), normalisation NFD, taux d'apprentissage 1e-4, lots de 16, augmentation d'images. L'entraînement (environ 10 min par époque sur T4) a été arrêté manuellement après l'époque 9.

Pour accélérer : ajouter `--workers 2 --precision 16-mixed` avant `train` (non utilisé pour les résultats publiés). Pour un arrêt automatique quand les gains deviennent négligeables : `--min-delta 0.002`.

In [ ]:
OUT = ROOT / "kraken_occitan"
OUT.mkdir(exist_ok=True)
!ketos -d cuda:0 train -f path -i "{MODEL_BASE}" --resize union -u NFD \
    -t train.txt -e dev.txt -o "{OUT}" -B 16 -r 0.0001 \
    --augment -q early --lag 10 --min-epochs 5 > entrainement.log 2>&1
!tail -n 5 entrainement.log

## 3. Conversion du meilleur checkpoint et évaluation

In [ ]:
# Les checkpoints sont nommés checkpoint_<époque>-<précision dev>.ckpt
ckpts = [c for c in OUT.glob("checkpoint_*-*.ckpt")]
meilleur = max(ckpts, key=lambda c: float(c.stem.split("-")[-1]))
print("Meilleur checkpoint :", meilleur.name)

MODEL_FT = OUT / "occitan_best.safetensors"
!ketos convert -o "{MODEL_FT}" "{meilleur}"

print("Kraken affiné :")
evaluer(DATA, MODEL_FT, "predictions_kraken_ft.csv")